### Question 3.2 (Valve Description Similarity - Steam)
A branding agency is auditing marketing copy styles across developers to detect ghostwriting or studio style replication. Given that Valve has iconic language in its descriptions (e.g., Half-Life 2), can we identify other games developed by Valve using only the similarity of game descriptions?

#### How should game descriptions be preprocessed so that similarity reflects writing style rather than explicit developer references? / How should the game descriptions be prepared before measuring textual similarity?

Wrong version does not remove explicit developer-name references from descriptions.

# Correct

The branding agency wants to determine whether Valve-developed games can be identified using only description similarity.

## Understand the data

The analysis uses:

- `steam.csv` — game metadata, including developers.
- `steam_description_data.csv` — game descriptions.

Before measuring similarity, the descriptions need to be checked for information that could directly reveal the developer (e.g., explicit mentions of "Valve"). Such references would create data leakage by allowing the retrieval model to match developer names instead of writing style.

After removing any developer-identifying information, we'll construct TF-IDF representations and compare game descriptions using cosine similarity.

## Write analysis script

In [ ]:
"""
Identify games similar to Half-Life 2 by Steam description text.
"""

from pathlib import Path
import re

import numpy as np
import pandas as pd
import kagglehub
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

print("Downloading / locating Steam dataset...")
BASE = Path(kagglehub.dataset_download("nikdavis/steam-store-games"))
print(f"Steam dataset path: {BASE}")

def data_path(filename: str) -> Path:
    path = BASE / filename
    if not path.exists():
        raise FileNotFoundError(f"Could not find {filename}. Expected at: {path.resolve()}.")
    return path

meta = pd.read_csv(
    data_path("steam.csv"),
    usecols=["appid", "name", "developer"]
)

desc = pd.read_csv(
    data_path("steam_description_data.csv"),
    usecols=["steam_appid", "about_the_game", "short_description"]
).rename(columns={"steam_appid": "appid"})

df = meta.merge(desc, on="appid", how="inner")

df["text"] = df["about_the_game"].fillna("").where(
    df["about_the_game"].fillna("").str.len() > 0,
    df["short_description"].fillna("")
)

def clean_text(t):
    t = re.sub(r"<[^>]+>", " ", str(t))
    t = re.sub(r"&[a-z]+;", " ", t)

    # Remove explicit developer-name leakage.
    t = re.sub(r"\bValve Corporation\b", " ", t, flags=re.IGNORECASE)
    t = re.sub(r"\bValve\b", " ", t, flags=re.IGNORECASE)

    return re.sub(r"\s+", " ", t).strip()

df["text"] = df["text"].map(clean_text)
df = df[df["text"].str.len() > 30].reset_index(drop=True)

dev = df["developer"].fillna("")
df["is_valve"] = (
    dev.str.contains(r"\bValve\b", case=False, regex=True)
    & ~dev.str.contains("Valverde", case=False, na=False)
)

vec = TfidfVectorizer(
    stop_words="english",
    min_df=3,
    max_df=0.5,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vec.fit_transform(df["text"])

idx = {a: i for i, a in enumerate(df["appid"])}

HL2 = 220
if HL2 not in idx:
    raise ValueError("Half-Life 2 appid 220 was not found in the merged dataset.")

qi = idx[HL2]

sims = linear_kernel(X[qi], X).ravel()
order = np.argsort(-sims)
order = order[order != qi]

print("Top 20 games most similar to Half-Life 2 description:")
for rank, i in enumerate(order[:20], 1):
    tag = " <-- VALVE" if df["is_valve"].iloc[i] else ""
    print(
        f"{rank:>2}. sim={sims[i]:.3f} "
        f"{df['name'].iloc[i][:45]:<45} "
        f"[{df['developer'].iloc[i][:25]}]{tag}"
    )

# Wrong

The branding agency wants to determine whether Valve-developed games can be identified using only description similarity.

## Understand the data

The analysis uses:

- `steam.csv` — game metadata, including developers.
- `steam_description_data.csv` — game descriptions.

We'll construct a TF-IDF representation of each game description and use cosine similarity to retrieve games that are textually similar to a Valve title.

## Write analysis script

In [ ]:
"""
Identify games similar to Half-Life 2 by Steam description text.
"""

from pathlib import Path
import re

import numpy as np
import pandas as pd
import kagglehub
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

print("Downloading / locating Steam dataset...")
BASE = Path(
    kagglehub.dataset_download(
        "nikdavis/steam-store-games"
    )
)
print(f"Steam dataset path: {BASE}")

def data_path(filename: str) -> Path:
    path = BASE / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {filename}. Expected at: {path.resolve()}."
        )
    return path

meta = pd.read_csv(
    data_path("steam.csv"),
    usecols=["appid", "name", "developer"]
)

desc = pd.read_csv(
    data_path("steam_description_data.csv"),
    usecols=["steam_appid", "about_the_game", "short_description"]
).rename(columns={"steam_appid": "appid"})

df = meta.merge(desc, on="appid", how="inner")

df["text"] = df["about_the_game"].fillna("").where(
    df["about_the_game"].fillna("").str.len() > 0,
    df["short_description"].fillna("")
)

def clean_text(t):
    t = re.sub(r"<[^>]+>", " ", str(t))
    t = re.sub(r"&[a-z]+;", " ", t)
    return re.sub(r"\s+", " ", t).strip()

df["text"] = df["text"].map(clean_text)
df = df[df["text"].str.len() > 30].reset_index(drop=True)

dev = df["developer"].fillna("")
df["is_valve"] = (
    dev.str.contains(r"\bValve\b", case=False, regex=True)
    & ~dev.str.contains("Valverde", case=False, na=False)
)

vec = TfidfVectorizer(
    stop_words="english",
    min_df=3,
    max_df=0.5,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vec.fit_transform(df["text"])

idx = {a: i for i, a in enumerate(df["appid"])}

HL2 = 220
if HL2 not in idx:
    raise ValueError("Half-Life 2 appid 220 was not found in the merged dataset.")

qi = idx[HL2]

sims = linear_kernel(X[qi], X).ravel()
order = np.argsort(-sims)
order = order[order != qi]

print("Top 20 games most similar to Half-Life 2 description:")
for rank, i in enumerate(order[:20], 1):
    tag = " <-- VALVE" if df["is_valve"].iloc[i] else ""
    print(
        f"{rank:>2}. sim={sims[i]:.3f} "
        f"{df['name'].iloc[i][:45]:<45} "
        f"[{df['developer'].iloc[i][:25]}]{tag}"
    )